In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier

In [2]:
df = pd.read_csv(
    "../../dataset/filtered_240_disease_dataset.csv"
)

print(df.shape)

(151556, 378)


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
X = df.drop("diseases", axis=1)

y = df["diseases"]

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

print("Number of diseases :", len(label_encoder.classes_))

Number of diseases : 240


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(136400, 377)
(15156, 377)


In [5]:
import catboost
print(catboost.__version__)

1.2.10


In [6]:
from catboost.utils import get_gpu_device_count

print("GPU Count:", get_gpu_device_count())

GPU Count: 1


In [7]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    task_type="GPU",
    devices="0",

    iterations=800,
    learning_rate=0.05,
    depth=8,

    loss_function="MultiClass",
    eval_metric="TotalF1",

    random_seed=42,

    early_stopping_rounds=100,

    verbose=50
)

cat_model.fit(

    X_train,

    y_train,

    eval_set=(X_test, y_test),

    use_best_model=True
)

0:	learn: 0.0449018	test: 0.0462688	best: 0.0462688 (0)	total: 1.2s	remaining: 16m 1s
50:	learn: 0.7192523	test: 0.7140052	best: 0.7140052 (50)	total: 1m 3s	remaining: 15m 28s
100:	learn: 0.8145339	test: 0.8106779	best: 0.8106779 (100)	total: 2m 6s	remaining: 14m 35s
150:	learn: 0.8423802	test: 0.8363280	best: 0.8363280 (150)	total: 3m 9s	remaining: 13m 32s
200:	learn: 0.8542867	test: 0.8485778	best: 0.8485778 (200)	total: 4m 12s	remaining: 12m 31s
250:	learn: 0.8591808	test: 0.8548028	best: 0.8549038 (245)	total: 5m 16s	remaining: 11m 32s
300:	learn: 0.8625558	test: 0.8572708	best: 0.8580263 (296)	total: 6m 24s	remaining: 10m 36s
350:	learn: 0.8647685	test: 0.8593658	best: 0.8596346 (349)	total: 7m 35s	remaining: 9m 42s
400:	learn: 0.8666638	test: 0.8609775	best: 0.8609775 (400)	total: 8m 42s	remaining: 8m 39s
450:	learn: 0.8680948	test: 0.8622279	best: 0.8622279 (450)	total: 9m 51s	remaining: 7m 37s
500:	learn: 0.8694342	test: 0.8613869	best: 0.8622279 (450)	total: 11m 1s	remaining: 

CatBoostClassifier(depth=8, devices='0', early_stopping_rounds=100, eval_metric='TotalF1', iterations=800, learning_rate=0.05, loss_function='MultiClass', random_seed=42, task_type='GPU', verbose=50)

In [8]:
pred = cat_model.predict(X_test)

pred = pred.astype(int).flatten()

In [9]:
acc = accuracy_score(y_test, pred)

macro = f1_score(
    y_test,
    pred,
    average="macro"
)

print("Accuracy :", acc)
print("Macro F1 :", macro)

Accuracy : 0.8630245447347585
Macro F1 : 0.8658252420679541


In [10]:
import numpy as np

# Select a test sample
sample = X_test.iloc[[0]]

# Predict probabilities
probs = cat_model.predict_proba(sample)[0]

# Number of top predictions
top_k = 5

# Get indices of top-k probabilities
top_indices = np.argsort(probs)[::-1][:top_k]

# Best prediction
best_idx = top_indices[0]
best_disease = label_encoder.inverse_transform([best_idx])[0]
best_confidence = probs[best_idx] * 100

# Confidence level
if best_confidence >= 90:
    confidence_level = "Very High"
elif best_confidence >= 75:
    confidence_level = "High"
elif best_confidence >= 50:
    confidence_level = "Moderate"
else:
    confidence_level = "Low"

print("=" * 70)
print(f"Predicted Disease : {best_disease}")
print(f"Confidence Score  : {best_confidence:.2f}%")
print(f"Confidence Level  : {confidence_level}")
print("=" * 70)

print("\nTop Predictions:\n")

for rank, idx in enumerate(top_indices, start=1):
    disease = label_encoder.inverse_transform([idx])[0]
    confidence = probs[idx] * 100

    print(f"{rank}. {disease:<45} {confidence:6.2f}%")

Predicted Disease : croup
Confidence Score  : 93.94%
Confidence Level  : Very High

Top Predictions:

1. croup                                          93.94%
2. acute bronchiolitis                             3.02%
3. acute bronchospasm                              1.12%
4. pneumonia                                       0.73%
5. otitis media                                    0.30%


In [11]:
print(
    classification_report(
        y_test,
        pred,
        target_names=label_encoder.classes_,
        digits=4
    )
)

                                                 precision    recall  f1-score   support

                               abdominal hernia     1.0000    0.9615    0.9804        26
                              actinic keratosis     0.9333    0.8642    0.8974        81
                            acute bronchiolitis     0.8837    0.9500    0.9157       120
                               acute bronchitis     0.8288    0.7603    0.7931       121
                             acute bronchospasm     0.7571    0.6543    0.7020        81
                            acute kidney injury     0.9405    0.9753    0.9576        81
                             acute otitis media     0.6780    0.7407    0.7080        54
                             acute pancreatitis     0.8429    0.9752    0.9042       121
     acute respiratory distress syndrome (ards)     0.7576    0.8333    0.7937        30
                                acute sinusitis     0.7500    0.9036    0.8197        83
                    

In [12]:
joblib.dump(cat_model, "./Models/catboost/catboost_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Model Saved Successfully.")

Model Saved Successfully.


In [ ]:
joblib.dump(X.columns.tolist(), "../Models/feature_columns.pkl")